# Edge colouring of complete tripartite graphs with z3

Cleaned, top-to-bottom runnable version. Every function is defined before it is used, and the graph is passed explicitly (no hidden global `x`).

In [1]:
import time
import pygraphviz as pgv
from z3 import *

## 1. Build a complete tripartite graph K(l, m, n)

`create_graph` returns the edge list; `build_graph` wraps it in a pygraphviz `AGraph`.

In [2]:
def create_graph(l, m, n):
    """Edges of the complete tripartite graph K(l, m, n)."""
    edges = []
    for i in range(0, l):
        for j in range(l, l + m):
            edges.append((i, j))
    for i in range(0, l):
        for k in range(l + m, l + m + n):
            edges.append((i, k))
    for j in range(l, l + m):
        for k in range(l + m, l + m + n):
            edges.append((j, k))
    return edges

In [3]:
def build_graph(l, m, n):
    """Build the AGraph for K(l, m, n). Returns (graph, name, layout)."""
    G = pgv.AGraph()
    G.node_attr['style'] = 'filled'
    G.edge_attr['dir'] = 'none'

    edges = create_graph(l, m, n)
    for i in range(l + m + n):
        G.add_node(i)
    for src, dst in edges:
        G.add_edge(src, dst)

    return (G, 'k_%d_%d_%d' % (l, m, n), 'circo')

## 2. Edge helpers

Edges are represented as `"a b"` strings. `all_edges` lists them; `node_nbs` returns every edge that shares an endpoint with the given edge (i.e. its neighbours in the line graph).

In [4]:
def all_edges(graph):
    """All edges of `graph` as 'a b' strings."""
    edges = []
    for i in range(graph.number_of_edges()):
        u, v = graph.edges()[i]
        edges.append(u + ' ' + v)
    return edges

In [5]:
def node_nbs(graph, edge):
    """Edges adjacent to `edge` (sharing either endpoint)."""
    neighbor_edges = []
    rev = edge.split(' ')[1] + ' ' + edge.split(' ')[0]   # true reverse edge (token swap)
    for j in range(2):
        node = edge.split(' ')[j]
        for nb in graph.neighbors(node):
            nb_edge = node + ' ' + nb
            if nb_edge != edge and nb_edge != rev:
                neighbor_edges.append(nb_edge)
    return neighbor_edges

## 3. Edge colouring with z3

Finds a minimum proper edge colouring. By Vizing's theorem the chromatic index is either Δ or Δ+1 (Δ = max degree), and Δ is a hard lower bound, so only those two counts are tested. Adjacency constraints are built once and deduplicated, and colour symmetry is broken by fixing the edges around a maximum-degree vertex.

In [6]:
def graph_coloring(graph):
    """Minimum edge colouring with z3.

    Optimised: the chromatic index is always Delta or Delta+1 (Vizing) with a
    hard lower bound of Delta, so we test only those two candidates instead of
    scanning from 2. The adjacency (!=) constraints are built once (deduped),
    and symmetry is broken by fixing the colours of the edges around a
    maximum-degree vertex. Returns {edge 'a b': colour}.
    """
    edges = all_edges(graph)                              # compute ONCE
    idx = {e: i for i, e in enumerate(edges)}
    xs = [Int('e%d' % i) for i in range(len(edges))]

    # line-graph adjacency, once, deduplicated (i < j)
    pairs = []
    for i, e in enumerate(edges):
        for nb in node_nbs(graph, e):
            j = idx.get(nb)
            if j is None:                                 # neighbour stored in the other orientation
                j = idx.get(nb.split(' ')[1] + ' ' + nb.split(' ')[0])
            if j is not None and i < j:
                pairs.append((i, j))

    delta = max((len(graph.neighbors(v)) for v in graph.nodes()), default=1)

    s = Solver()
    for i, j in pairs:
        s.add(xs[i] != xs[j])                             # != constraints added ONCE

    # symmetry breaking: fix colours of the edges incident to a max-degree vertex
    hub = max(graph.nodes(), key=lambda v: len(graph.neighbors(v)))
    hub_edges = [i for i, e in enumerate(edges) if hub in e.split(' ')]
    for c, i in enumerate(hub_edges):
        s.add(xs[i] == c)

    for k in (delta, delta + 1):                          # only two candidates (Vizing)
        s.push()
        for x in xs:
            s.add(x >= 0, x <= k - 1)
        if s.check() == sat:
            print('OK, found a solution with %d colors' % k)
            m = s.model()
            return {e: m[xs[idx[e]]].as_long() for e in edges}
        s.pop()

    raise Exception('Could not find a solution.')         # unreachable for simple graphs

## 3b. Solution → matrix

`color_matrix` turns an edge-colour solution into a symmetric n×n matrix (`M[u][v]` = colour of edge (u,v)); `*` marks cells with no edge, including the diagonal. `format_matrix` renders it as aligned text for saving/printing.

In [7]:
def color_matrix(graph, solution):
    """Symmetric n x n edge-colour matrix; '*' where there is no edge."""
    nodes = sorted(graph.nodes(), key=int)
    idx = {node: i for i, node in enumerate(nodes)}
    M = [['*'] * len(nodes) for _ in nodes]
    for e_str, color in solution.items():
        u, v = e_str.split(' ')
        M[idx[u]][idx[v]] = str(color)
        M[idx[v]][idx[u]] = str(color)
    return nodes, M


def format_matrix(nodes, M):
    """Render the matrix as aligned text with node labels on row/col."""
    w = max([len(x) for row in M for x in row] + [len(x) for x in nodes]) + 1
    lines = [' ' * w + ''.join(x.rjust(w) for x in nodes)]
    for label, row in zip(nodes, M):
        lines.append(label.rjust(w) + ''.join(x.rjust(w) for x in row))
    return '\n'.join(lines)


def format_kmn(name, solution):
    """Screenshot-style matrix for K(l, m, n).

    Rows = part A (size l) then part B (size m); columns = part A (size l)
    then part C (size n). '*' marks no-edge cells (the A x A block). Cells are
    separated by ' | ' with an underline under each row, titled 'K l m n'.
    """
    l, m, n = (int(t) for t in name.split('_')[1:])
    row_nodes = [str(v) for v in range(l + m)]                              # A, then B
    col_nodes = [str(v) for v in list(range(l)) + list(range(l + m, l + m + n))]  # A, then C

    def cell(u, v):
        c = solution.get(u + ' ' + v)
        if c is None:
            c = solution.get(v + ' ' + u)
        return '*' if c is None else str(c)

    grid = [[cell(r, c) for c in col_nodes] for r in row_nodes]
    w = max(len(x) for row in grid for x in row)
    rows_txt = [' ' + ' | '.join(x.rjust(w) for x in row) + ' |' for row in grid]
    width = max(len(r) for r in rows_txt)
    rule = ' ' + '-' * (width - 1)

    lines = [('K ' + ' '.join(str(s) for s in (l, m, n))).center(width), '']
    for r in rows_txt:
        lines.append(r)
        lines.append(rule)
    return '\n'.join(lines)


def render_matrix(name, graph, solution):
    """K(l,m,n) screenshot layout when the name is 'k_l_m_n', else plain matrix."""
    parts = name.split('_')
    if parts[0] == 'k' and len(parts) == 4 and all(p.isdigit() for p in parts[1:]):
        return format_kmn(name, solution)
    nodes, M = color_matrix(graph, solution)
    return format_matrix(nodes, M)

## 4. Run

Build K(1, 1, 1) (a triangle) and colour its edges. The three mutually-adjacent edges each need a different colour, so 3 colours.

In [8]:
x, name, layout = build_graph(2, 3, 9)

print('Graph %r: %d nodes, %d edges' % (name, x.number_of_nodes(), x.number_of_edges()))
print('edges     :', all_edges(x))
print("neighbours of '0 1':", node_nbs(x, '0 1'))

t1 = time.time()
solution = graph_coloring(x)
t2 = time.time()

print('solution (in %.3fs):' % (t2 - t1))
print(solution)

Graph 'k_2_3_9': 14 nodes, 51 edges
edges     : ['0 2', '0 3', '0 4', '0 5', '0 6', '0 7', '0 8', '0 9', '0 10', '0 11', '0 12', '0 13', '1 2', '1 3', '1 4', '1 5', '1 6', '1 7', '1 8', '1 9', '1 10', '1 11', '1 12', '1 13', '2 5', '2 6', '2 7', '2 8', '2 9', '2 10', '2 11', '2 12', '2 13', '3 5', '3 6', '3 7', '3 8', '3 9', '3 10', '3 11', '3 12', '3 13', '4 5', '4 6', '4 7', '4 8', '4 9', '4 10', '4 11', '4 12', '4 13']
neighbours of '0 1': ['0 2', '0 3', '0 4', '0 5', '0 6', '0 7', '0 8', '0 9', '0 10', '0 11', '0 12', '0 13', '1 2', '1 3', '1 4', '1 5', '1 6', '1 7', '1 8', '1 9', '1 10', '1 11', '1 12', '1 13']
OK, found a solution with 12 colors
solution (in 0.128s):
{'0 2': 0, '0 3': 1, '0 4': 2, '0 5': 3, '0 6': 4, '0 7': 5, '0 8': 6, '0 9': 7, '0 10': 8, '0 11': 9, '0 12': 10, '0 13': 11, '1 2': 5, '1 3': 3, '1 4': 8, '1 5': 11, '1 6': 10, '1 7': 1, '1 8': 2, '1 9': 0, '1 10': 6, '1 11': 7, '1 12': 4, '1 13': 9, '2 5': 2, '2 6': 3, '2 7': 10, '2 8': 1, '2 9': 6, '2 10': 4, '2 

## 5. Draw the coloured graph and save the matrix

Saves the edge-colour solution as `<name>_solution.txt` (the n×n matrix) and draws `<name>_edge_colored.png`. Both filenames come from the graph's `name`, so this cell works for whatever graph you built above — nothing is hardcoded to a specific graph.

In [9]:
# Draw the coloured graph and save the edge-colour solution matrix.
# Filenames are derived from the graph's name, so this works for ANY graph.
color_available = ['red', 'blue', 'green', 'pink', 'orange', 'purple', 'brown', 'gray']

# save the solution matrix (screenshot layout for K(l,m,n))
text = render_matrix(name, x, solution)
print(text)
with open('%s_solution.txt' % name, 'w') as f:
    f.write(text + '\n')
print('Saved %s_solution.txt' % name)

# colour the edges and draw  --- IMAGE GENERATION DISABLED FOR NOW ---
# for e_str, color in solution.items():
#     u, v = e_str.split(' ')
#     x.get_edge(u, v).attr['color'] = color_available[color % len(color_available)]
# x.layout('dot')
# x.draw('%s_edge_colored.png' % name)
# print('Saved %s_edge_colored.png' % name)

                        K 2 3 9                        

  * |  * |  3 |  4 |  5 |  6 |  7 |  8 |  9 | 10 | 11 |
 ------------------------------------------------------
  * |  * | 11 | 10 |  1 |  2 |  0 |  6 |  7 |  4 |  9 |
 ------------------------------------------------------
  0 |  5 |  2 |  3 | 10 |  1 |  6 |  4 |  8 |  9 |  7 |
 ------------------------------------------------------
  1 |  3 |  0 |  2 |  6 |  4 | 10 | 11 |  5 |  7 |  8 |
 ------------------------------------------------------
  2 |  8 | 10 |  9 |  4 |  3 |  5 |  0 |  1 | 11 |  6 |
 ------------------------------------------------------
Saved k_2_3_9_solution.txt


## 6. `main()`: colour + minimum dominating set

Restores the functions the original `main()` needed (they were commented out): `min_dom_set` and `build_peternson_3_coloring_graph`. `main()` edge-colours each graph (using the `graph_coloring` above) and then finds a minimum dominating set, saving a PNG for each.

> `build_fat_graph()` is omitted: it reads `graph_coloring_z3_example_fat_graph.gv`, which is not present.
> The original also used `G.nodes_iter()`, removed in modern pygraphviz; `G.nodes()` is used instead.

In [10]:
def build_peternson_3_coloring_graph():
    """Petersen graph (en.wikipedia.org/wiki/File:Petersen_graph_3-coloring.svg)."""
    G = pgv.AGraph()
    G.node_attr['style'] = 'filled'
    G.edge_attr['dir'] = 'none'

    edges = [
        (0, 2), (0, 1), (0, 5), (0, 4), (1, 6), (1, 7),
        (2, 3), (2, 8), (3, 4), (3, 7), (4, 5), (4, 6),
        (5, 9), (6, 8), (7, 9), (8, 9), (9, 3),
    ]
    for i in range(10):
        G.add_node(i)
    for src, dst in edges:
        G.add_edge(src, dst)

    return (G, 'peternson_3_coloring_graph', 'circo')

In [11]:
# Contributed by @cbrewbs chad@dataculture.co  (requires z3 >= 4.4.1)
def min_dom_set(graph):
    """Dominate the graph with the fewest vertices possible. Returns {node: 0/1}."""
    s = Optimize()
    nodes_colors = dict((node, Int('k%r' % node)) for node in graph.nodes())
    for node in graph.nodes():
        s.add(And(nodes_colors[node] >= 0, nodes_colors[node] <= 1))  # dominator or not
        dom_neighbor = Sum([nodes_colors[j] for j in graph.neighbors(node)])
        s.add(Sum(nodes_colors[node], dom_neighbor) >= 1)
    s.minimize(Sum([nodes_colors[y] for y in graph.nodes()]))

    if s.check() == sat:
        m = s.model()
        return dict((name, m[color].as_long()) for name, color in nodes_colors.items())

    raise Exception('Could not find a solution.')

In [12]:
def main(Gs, matrix_path='solutions.txt'):
    """Edge-colour and find a min dominating set for each (graph, name, layout).

    Edge-colour solutions are also saved as n x n matrices ('*' = no edge)
    into a single text file `matrix_path`, one section per graph.
    """
    color_available = ['red', 'blue', 'green', 'pink', 'orange', 'purple', 'brown', 'gray']

    with open(matrix_path, 'w') as fout:
        for G, name, layout in Gs:
            print('Trying to color %r now (%d nodes, %d edges)..'
                  % (name, G.number_of_nodes(), G.number_of_edges()))
            t1 = time.time()
            s = graph_coloring(G)          # {edge 'a b': colour}
            t2 = time.time()
            print('Here is the solution (in %.3fs):' % (t2 - t1))
            print(s if len(s) < 20 else 'Too long, see the .png!')

            fout.write(render_matrix(name, G, s) + '\n\n')

            # --- IMAGE GENERATION DISABLED FOR NOW ---
            # for e_str, color in s.items():
            #     u, v = e_str.split(' ')
            #     G.get_edge(u, v).attr['color'] = color_available[color % len(color_available)]
            # G.layout(layout)
            # G.draw('graph_coloring_z3_%s_colored.png' % name)
            # print('Saved graph_coloring_z3_%s_colored.png' % name)
            print('---')
    print('Saved edge-colour matrices to %s' % matrix_path)
    print('===')

    for G, name, layout in Gs:
        print('Trying to find min dom set for %r now (%d nodes, %d edges)..'
              % (name, G.number_of_nodes(), G.number_of_edges()))
        t1 = time.time()
        s = min_dom_set(G)             # {node: 0/1}
        t2 = time.time()
        print('OK, found a solution with %d dominators (in %.3fs):'
              % (sum(s.values()), (t2 - t1)))
        print(s if len(s) < 20 else 'Too long, see the .png!')

        # --- IMAGE GENERATION DISABLED FOR NOW ---
        # for node in G.nodes():
        #     G.get_node(node).attr['fillcolor'] = color_available[s[node]]  # dominators in blue
        # G.layout(layout)
        # G.draw('graph_z3_%s_dominated.png' % name)
        # print('Saved graph_z3_%s_dominated.png' % name)
        print('---')

    return 1

## 7. Run `main()` on some small graphs

In [13]:
# Edit PARAMS to run on any complete tripartite graphs K(l, m, n).
PARAMS = [
    (1, 1, 1),
    (1, 2, 3),
    (2, 2, 2),
]

main([build_graph(l, m, n) for (l, m, n) in PARAMS])

Trying to color 'k_1_1_1' now (3 nodes, 3 edges)..
OK, found a solution with 3 colors
Here is the solution (in 0.002s):
{'0 1': 0, '0 2': 1, '1 2': 2}
---
Trying to color 'k_1_2_3' now (6 nodes, 11 edges)..
OK, found a solution with 5 colors
Here is the solution (in 0.005s):
{'0 1': 0, '0 2': 1, '0 3': 2, '0 4': 3, '0 5': 4, '1 3': 3, '1 4': 4, '1 5': 1, '2 3': 0, '2 4': 2, '2 5': 3}
---
Trying to color 'k_2_2_2' now (6 nodes, 12 edges)..
OK, found a solution with 4 colors
Here is the solution (in 0.005s):
{'0 2': 0, '0 3': 1, '0 4': 2, '0 5': 3, '1 2': 3, '1 3': 2, '1 4': 0, '1 5': 1, '2 4': 1, '2 5': 2, '3 4': 3, '3 5': 0}
---
Saved edge-colour matrices to solutions.txt
===
Trying to find min dom set for 'k_1_1_1' now (3 nodes, 3 edges)..
OK, found a solution with 1 dominators (in 0.011s):
{'0': 0, '1': 1, '2': 0}
---
Trying to find min dom set for 'k_1_2_3' now (6 nodes, 11 edges)..
OK, found a solution with 1 dominators (in 0.003s):
{'0': 1, '1': 0, '2': 0, '3': 0, '4': 0, '5': 0}


1